# LayoutLMv3 Invoice Processor - Google Colab

## Sistema completo para procesar 34,165 facturas peruanas

Este notebook procesa facturas PDF y las convierte al formato LayoutLMv3 (FUNSD-style).

### Dataset:
- **Ubicación**: `/content/drive/MyDrive/dataset_entrenamiento_101125/`
- **PDFs**: `facturas_pdf/` - 34,165 archivos
- **JSONs**: `anotaciones_json/` - 34,165 archivos

### Tiempo estimado:
- 1 documento: ~8 segundos
- 1,000 documentos: ~2.2 horas
- 34,165 documentos: ~75 horas (usar en lotes)

---

## 📋 PASO 1: Configuración Inicial

### Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Verificar que el dataset existe

In [ ]:
import os

# Ruta al dataset
DATASET_DIR = "/content/drive/MyDrive/dataset_entrenamiento_101125"
PDF_DIR = f"{DATASET_DIR}/facturas_pdf"
JSON_DIR = f"{DATASET_DIR}/anotaciones_json"

# Verificar
print(f"📁 Dataset existe: {os.path.exists(DATASET_DIR)}")
print(f"📄 PDFs existen: {os.path.exists(PDF_DIR)}")
print(f"📋 JSONs existen: {os.path.exists(JSON_DIR)}")

if os.path.exists(PDF_DIR):
    pdf_count = len([f for f in os.listdir(PDF_DIR) if f.endswith('.pdf')])
    print(f"\n✅ Total de PDFs encontrados: {pdf_count}")

if os.path.exists(JSON_DIR):
    json_count = len([f for f in os.listdir(JSON_DIR) if f.endswith('.json')])
    print(f"✅ Total de JSONs encontrados: {json_count}")

## 🔧 PASO 2: Instalar Dependencias

In [ ]:
# Instalar Tesseract OCR
!apt-get update -qq
!apt-get install -y tesseract-ocr tesseract-ocr-spa

# Verificar instalación
!tesseract --version

## 📥 PASO 3: Clonar Repositorio e Instalar Paquetes Python

In [ ]:
# Ir al directorio de trabajo
%cd /content

# Clonar repositorio
!git clone https://github.com/GynoRomeroPrado/extraccion-coordenadas-OCR.git
%cd extraccion-coordenadas-OCR

# Checkout a la rama con el código
!git checkout claude/layoutlmv3-invoice-processor-011CV2g24oh8fxvVVWutKLKF

In [ ]:
# Instalar dependencias Python
!pip install -q -r requirements.txt

print("✅ Dependencias instaladas")

## 🧪 PASO 4: Probar con UN Documento

Primero probamos con un solo documento para verificar que todo funciona.

In [ ]:
# Crear directorio de salida
OUTPUT_DIR = "/content/drive/MyDrive/output_layoutlmv3"
!mkdir -p {OUTPUT_DIR}

# Procesar primer documento
!python scripts/process_single.py \
    --pdf "{PDF_DIR}/FACT-000001.pdf" \
    --json "{JSON_DIR}/FACT-000001.json" \
    --output "{OUTPUT_DIR}" \
    --verbose \
    --validate

### Verificar resultado

In [ ]:
import json

# Leer metadata
metadata_file = f"{OUTPUT_DIR}/FACT-000001_metadata.json"

if os.path.exists(metadata_file):
    with open(metadata_file, 'r', encoding='utf-8') as f:
        metadata = json.load(f)
    
    print("📊 RESULTADOS DEL DOCUMENTO DE PRUEBA:")
    print("=" * 60)
    print(f"📄 Documento: {metadata['document_id']}")
    print(f"📄 Páginas: {metadata['num_pages']}")
    print(f"🔄 Orientación detectada: {metadata['orientation_detected']}°")
    print(f"⚙️  Método de extracción: {metadata['extraction_method']}")
    print(f"\n📊 Header:")
    header = metadata['components']['header']
    print(f"   Campos encontrados: {header['fields_found']}/{header['fields_total']}")
    print(f"   Match rate: {header['match_rate']:.1%}")
    print(f"\n📦 Items:")
    items = metadata['components']['items']
    print(f"   Items: {items['items_total']}")
    if items['match_rate']:
        print(f"   Match rate: {items['match_rate']:.1%}")
    print(f"\n⏱️  Tiempo: {metadata['statistics']['processing_time_seconds']:.2f}s")
    print(f"💾 Confianza promedio: {metadata['statistics']['confidence_avg']:.1%}")
else:
    print("❌ No se encontró el archivo de metadata")

## 🚀 PASO 5: Procesamiento en Lote

### Opción A: Procesar los primeros 100 documentos (para prueba rápida)

In [ ]:
# Procesar primeros 100 documentos
!python scripts/process_batch.py \
    --input-dir "{DATASET_DIR}" \
    --output-dir "{OUTPUT_DIR}" \
    --start-index 0 \
    --end-index 100 \
    --skip-existing \
    --checkpoint-interval 10

### Opción B: Procesar los primeros 1,000 documentos

In [ ]:
# Procesar primeros 1,000 documentos (~2.2 horas)
!python scripts/process_batch.py \
    --input-dir "{DATASET_DIR}" \
    --output-dir "{OUTPUT_DIR}" \
    --start-index 0 \
    --end-index 1000 \
    --skip-existing \
    --checkpoint-interval 100

### Opción C: Procesar TODOS los documentos en lotes

**IMPORTANTE**: Google Colab tiene límite de tiempo. Recomendado procesarlos en lotes de 1,000.

Ejecutar este código múltiples veces cambiando el lote:

In [ ]:
# CONFIGURAR EL LOTE A PROCESAR
BATCH_NUMBER = 0  # Cambiar a 0, 1, 2, 3, ... 34
BATCH_SIZE = 1000

start_index = BATCH_NUMBER * BATCH_SIZE
end_index = (BATCH_NUMBER + 1) * BATCH_SIZE

print(f"🚀 Procesando lote {BATCH_NUMBER}")
print(f"📄 Documentos: {start_index} a {end_index}")
print("=" * 60)

!python scripts/process_batch.py \
    --input-dir "{DATASET_DIR}" \
    --output-dir "{OUTPUT_DIR}" \
    --start-index {start_index} \
    --end-index {end_index} \
    --skip-existing \
    --checkpoint-interval 100

## 📊 PASO 6: Ver Estadísticas del Procesamiento

In [ ]:
# Leer estadísticas del batch
stats_file = f"{OUTPUT_DIR}/batch_statistics.json"

if os.path.exists(stats_file):
    with open(stats_file, 'r', encoding='utf-8') as f:
        stats = json.load(f)
    
    print("📊 ESTADÍSTICAS DEL BATCH")
    print("=" * 60)
    print(f"📄 Total de documentos: {stats['total_documents']}")
    print(f"✅ Procesados: {stats['processed']} ({stats['success_rate']:.1%})")
    print(f"❌ Fallidos: {stats['failed']}")
    print(f"⏭️  Saltados: {stats['skipped']}")
    print(f"⏱️  Tiempo total: {stats['total_processing_time']:.2f}s ({stats['total_processing_time']/60:.1f} min)")
    print(f"⏱️  Tiempo promedio: {stats['avg_processing_time']:.2f}s/doc")
    print(f"📊 Match rate promedio (header): {stats['avg_match_rate_header']:.1%}")
    if stats['avg_match_rate_items']:
        print(f"📊 Match rate promedio (items): {stats['avg_match_rate_items']:.1%}")
    
    if stats['failed_documents']:
        print(f"\n⚠️  Documentos fallidos ({len(stats['failed_documents'])}):")
        for doc in stats['failed_documents'][:10]:  # Mostrar primeros 10
            print(f"   • {doc}")
else:
    print("⚠️  No se encontraron estadísticas del batch")

## ✅ PASO 7: Validar Dataset Generado

In [ ]:
# Validar todos los documentos generados
!python scripts/validate_output.py \
    --output-dir "{OUTPUT_DIR}" \
    --check-metadata \
    --save-report "{OUTPUT_DIR}/validation_report.json"

### Ver reporte de validación

In [ ]:
# Leer reporte de validación
report_file = f"{OUTPUT_DIR}/validation_report.json"

if os.path.exists(report_file):
    with open(report_file, 'r', encoding='utf-8') as f:
        report = json.load(f)
    
    summary = report['summary']
    
    print("📊 REPORTE DE VALIDACIÓN")
    print("=" * 60)
    print(f"📄 Total de documentos: {summary['total_documents']}")
    print(f"✅ Válidos: {summary['valid']} ({summary['success_rate']:.1%})")
    print(f"❌ Inválidos: {summary['invalid']}")
    print(f"⚠️  Total de warnings: {summary['total_warnings']}")
    print(f"❌ Total de errores: {summary['total_errors']}")
    
    if summary['invalid'] > 0:
        print(f"\n⚠️  Primeros errores:")
        for error in report['all_errors'][:5]:
            print(f"   • {error}")
else:
    print("⚠️  No se encontró el reporte de validación")

## 📁 PASO 8: Ver Ejemplos de Documentos Generados

In [ ]:
# Listar archivos generados
import glob

json_files = glob.glob(f"{OUTPUT_DIR}/*.json")
layoutlmv3_files = [f for f in json_files if not f.endswith('_metadata.json') 
                    and 'batch_statistics' not in f 
                    and 'validation_report' not in f
                    and 'checkpoint' not in f]

print(f"📁 Archivos LayoutLMv3 generados: {len(layoutlmv3_files)}")
print(f"\nPrimeros 10 archivos:")
for f in sorted(layoutlmv3_files)[:10]:
    print(f"   • {os.path.basename(f)}")

### Ver contenido de un documento generado

In [ ]:
# Ver primer documento generado
if layoutlmv3_files:
    sample_file = sorted(layoutlmv3_files)[0]
    
    with open(sample_file, 'r', encoding='utf-8') as f:
        document = json.load(f)
    
    print(f"📄 Archivo: {os.path.basename(sample_file)}")
    print("=" * 60)
    print(f"Total de elementos: {len(document['form'])}")
    print(f"\nPrimeros 3 elementos:")
    print(json.dumps(document['form'][:3], indent=2, ensure_ascii=False))

## 📊 PASO 9: Análisis del Dataset Completo

In [ ]:
# Analizar dataset completo
import pandas as pd

metadata_files = glob.glob(f"{OUTPUT_DIR}/*_metadata.json")

print(f"📊 Analizando {len(metadata_files)} documentos procesados...\n")

# Recolectar estadísticas
data = []

for meta_file in metadata_files:
    with open(meta_file, 'r', encoding='utf-8') as f:
        meta = json.load(f)
    
    data.append({
        'document_id': meta['document_id'],
        'num_pages': meta.get('num_pages', 0),
        'orientation': meta.get('orientation_detected', 0),
        'method': meta.get('extraction_method', 'unknown'),
        'header_match_rate': meta['components']['header'].get('match_rate', 0),
        'items_match_rate': meta['components']['items'].get('match_rate', 0),
        'processing_time': meta['statistics'].get('processing_time_seconds', 0),
        'confidence': meta['statistics'].get('confidence_avg', 0)
    })

df = pd.DataFrame(data)

print("📊 ESTADÍSTICAS GENERALES")
print("=" * 60)
print(f"Total de documentos: {len(df)}")
print(f"\n📄 Distribución por número de páginas:")
print(df['num_pages'].value_counts().sort_index())
print(f"\n🔄 Distribución por orientación:")
print(df['orientation'].value_counts().sort_index())
print(f"\n⚙️  Distribución por método de extracción:")
print(df['method'].value_counts())
print(f"\n📊 Promedios:")
print(f"   Match rate (header): {df['header_match_rate'].mean():.1%}")
print(f"   Match rate (items): {df['items_match_rate'].mean():.1%}")
print(f"   Tiempo de procesamiento: {df['processing_time'].mean():.2f}s")
print(f"   Confianza promedio: {df['confidence'].mean():.1%}")

## 📊 PASO 10: Visualizaciones

In [ ]:
import matplotlib.pyplot as plt

# Crear visualizaciones
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Distribución de match rate (header)
axes[0, 0].hist(df['header_match_rate'], bins=20, edgecolor='black')
axes[0, 0].set_title('Distribución de Match Rate (Header)')
axes[0, 0].set_xlabel('Match Rate')
axes[0, 0].set_ylabel('Frecuencia')
axes[0, 0].axvline(df['header_match_rate'].mean(), color='red', linestyle='--', label='Promedio')
axes[0, 0].legend()

# 2. Distribución de match rate (items)
axes[0, 1].hist(df['items_match_rate'].dropna(), bins=20, edgecolor='black')
axes[0, 1].set_title('Distribución de Match Rate (Items)')
axes[0, 1].set_xlabel('Match Rate')
axes[0, 1].set_ylabel('Frecuencia')
axes[0, 1].axvline(df['items_match_rate'].mean(), color='red', linestyle='--', label='Promedio')
axes[0, 1].legend()

# 3. Distribución de tiempos de procesamiento
axes[1, 0].hist(df['processing_time'], bins=20, edgecolor='black')
axes[1, 0].set_title('Distribución de Tiempos de Procesamiento')
axes[1, 0].set_xlabel('Tiempo (segundos)')
axes[1, 0].set_ylabel('Frecuencia')
axes[1, 0].axvline(df['processing_time'].mean(), color='red', linestyle='--', label='Promedio')
axes[1, 0].legend()

# 4. Confianza promedio
axes[1, 1].hist(df['confidence'], bins=20, edgecolor='black')
axes[1, 1].set_title('Distribución de Confianza Promedio')
axes[1, 1].set_xlabel('Confianza')
axes[1, 1].set_ylabel('Frecuencia')
axes[1, 1].axvline(df['confidence'].mean(), color='red', linestyle='--', label='Promedio')
axes[1, 1].legend()

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/statistics_visualization.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\n💾 Visualización guardada en: {OUTPUT_DIR}/statistics_visualization.png")

## 💾 PASO 11: Descargar Resultados

### Opción A: Descargar como ZIP

In [ ]:
# Crear ZIP con todos los resultados
import shutil

zip_file = "/content/drive/MyDrive/output_layoutlmv3_backup"
shutil.make_archive(zip_file, 'zip', OUTPUT_DIR)

print(f"✅ ZIP creado: {zip_file}.zip")
print(f"📦 Tamaño: {os.path.getsize(zip_file + '.zip') / (1024*1024):.2f} MB")

### Opción B: Los archivos ya están en tu Google Drive

Los resultados ya están guardados en:
```
/content/drive/MyDrive/output_layoutlmv3/
```

Puedes acceder a ellos directamente desde Google Drive.

## 🔧 UTILS: Comandos Útiles

In [ ]:
# Ver checkpoint actual
checkpoint_file = f"{OUTPUT_DIR}/checkpoint.json"
if os.path.exists(checkpoint_file):
    with open(checkpoint_file, 'r') as f:
        checkpoint = json.load(f)
    print("📍 CHECKPOINT ACTUAL")
    print(f"   Timestamp: {checkpoint['timestamp']}")
    print(f"   Procesados: {checkpoint['processed']}")
    print(f"   Fallidos: {checkpoint['failed']}")
    print(f"   Progreso: {checkpoint['progress']}%")
else:
    print("⚠️  No hay checkpoint guardado")

In [ ]:
# Ver uso de disco
!du -sh {OUTPUT_DIR}
!df -h | grep drive

In [ ]:
# Limpiar archivos temporales
import glob

temp_files = glob.glob("/tmp/*_rotated_*.pdf")
if temp_files:
    for f in temp_files:
        os.remove(f)
    print(f"🧹 {len(temp_files)} archivos temporales eliminados")
else:
    print("✅ No hay archivos temporales")

## 📝 Notas Importantes

### Límites de Google Colab:
- **Tiempo máximo**: 12 horas por sesión
- **RAM**: 12-15 GB
- **Disco**: ~100 GB

### Recomendaciones:
1. **Procesar en lotes de 1,000** documentos por sesión
2. **Usar `--skip-existing`** para continuar si se interrumpe
3. **Guardar en Google Drive** para persistencia
4. **Monitorear checkpoints** regularmente
5. **Validar** después de cada lote grande

### Si la sesión se interrumpe:
1. Volver a ejecutar las celdas de configuración (Pasos 1-3)
2. El sistema continuará desde el último checkpoint
3. Los archivos en Google Drive no se pierden

---

## ✅ ¡Listo para procesar 34,165 facturas!

Ejecuta las celdas en orden y el sistema procesará todo el dataset automáticamente.